# Simple CNN for Face Recognition

In this notebook, we will build a **Convolutional Neural Network (CNN)** from scratch using **TensorFlow/Keras** to recognize faces.

## What is a CNN?
A CNN is a type of deep learning model specifically designed for processing **image data**. It uses:
- **Convolutional layers** – to detect features like edges, textures, and shapes
- **Pooling layers** – to reduce spatial dimensions and computation
- **Dense (fully connected) layers** – to make final classification decisions

## Dataset
We will use the **Labeled Faces in the Wild (LFW)** dataset, available directly through `scikit-learn`. This dataset contains face photographs of public figures collected from the internet.

## Workflow
1. Load and explore the dataset
2. Preprocess the images
3. Build a CNN model
4. Train the model
5. Evaluate performance
6. Visualize predictions

## Step 1: Import Libraries

In [ ]:
# Core libraries
import numpy as np
import matplotlib.pyplot as plt

# Scikit-learn – dataset and utilities
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# TensorFlow / Keras – deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")

## Step 2: Load the Dataset

We use `fetch_lfw_people` from scikit-learn which downloads the **Labeled Faces in the Wild** dataset.

We set `min_faces_per_person=70` so we only keep people who have **at least 70 photos** — this gives us enough training data per class.

In [ ]:
# Load the LFW dataset (only people with at least 70 images)
lfw_people = fetch_lfw_people(min_faces_per_person=70, resize=0.4)

# Extract images and labels
X = lfw_people.images        # shape: (n_samples, height, width)
y = lfw_people.target         # integer labels
target_names = lfw_people.target_names  # actual names

n_samples, h, w = X.shape
n_classes = len(target_names)

print(f"Total images:   {n_samples}")
print(f"Image size:     {h} x {w} pixels")
print(f"Number of people (classes): {n_classes}")
print(f"\nPeople in dataset:")
for i, name in enumerate(target_names):
    count = np.sum(y == i)
    print(f"  {name}: {count} images")

### Visualize Sample Faces

Let's look at some example images from the dataset to understand what we're working with.

In [ ]:
# Display a grid of sample faces
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i], cmap='gray')
    ax.set_title(target_names[y[i]], fontsize=10)
    ax.axis('off')
plt.suptitle("Sample Faces from LFW Dataset", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3: Preprocess the Data

Before feeding images into a CNN, we need to:

1. **Normalize pixel values** – Scale from `[0, 255]` to `[0, 1]` so the network trains faster
2. **Reshape for CNN** – Keras expects 4D input: `(samples, height, width, channels)`. Since these are grayscale images, `channels = 1`
3. **One-hot encode labels** – Convert integer labels to binary vectors (e.g., class 2 → `[0, 0, 1, 0, 0, ...]`)
4. **Split into train/test sets** – 80% training, 20% testing

In [ ]:
# 1. Normalize pixel values to [0, 1]
X_normalized = X / 255.0

# 2. Reshape for CNN: (samples, height, width, 1) — 1 channel for grayscale
X_reshaped = X_normalized.reshape(n_samples, h, w, 1)

# 3. One-hot encode the labels
y_encoded = to_categorical(y, num_classes=n_classes)

# 4. Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X_reshaped, y_encoded, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:  {X_train.shape[0]} images")
print(f"Testing set:   {X_test.shape[0]} images")
print(f"Input shape:   {X_train.shape[1:]}")
print(f"Output classes: {n_classes}")

## Step 4: Build the CNN Model

Our CNN architecture:

| Layer | Type | Details |
|-------|------|---------|
| 1 | Conv2D | 32 filters, 3×3 kernel, ReLU activation |
| 2 | BatchNormalization | Stabilizes training |
| 3 | MaxPooling2D | 2×2 pool size — reduces spatial dimensions |
| 4 | Conv2D | 64 filters, 3×3 kernel, ReLU activation |
| 5 | BatchNormalization | Stabilizes training |
| 6 | MaxPooling2D | 2×2 pool size |
| 7 | Conv2D | 128 filters, 3×3 kernel, ReLU activation |
| 8 | BatchNormalization | Stabilizes training |
| 9 | MaxPooling2D | 2×2 pool size |
| 10 | Flatten | Converts 2D feature maps to 1D vector |
| 11 | Dense | 256 neurons, ReLU activation |
| 12 | Dropout | 50% — prevents overfitting |
| 13 | Dense (output) | `n_classes` neurons, softmax activation |

In [ ]:
# Build the CNN model
model = Sequential([
    # --- Block 1 ---
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(h, w, 1)),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    # --- Block 2 ---
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    # --- Block 3 ---
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    # --- Classifier Head ---
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(n_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()

## Step 5: Train the Model

We train for up to **30 epochs** with **EarlyStopping** — if validation accuracy doesn't improve for 5 consecutive epochs, training stops automatically. This prevents overfitting and saves time.

We use a **validation split of 20%** from the training set to monitor performance during training.

In [ ]:
# Set up early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

### Training Curves

Plotting training vs. validation **accuracy** and **loss** helps us check for overfitting. If training accuracy keeps rising while validation accuracy plateaus or drops, the model is overfitting.

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(history.history['loss'], label='Training Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Evaluate the Model

Let's evaluate the trained model on the **test set** (data the model has never seen during training).

We'll look at:
- **Test accuracy** — overall percentage of correct predictions
- **Classification report** — precision, recall, and F1-score per person
- **Confusion matrix** — shows where the model confuses one person for another

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Evaluate on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test Loss:     {test_loss:.4f}")

# Get predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Classification Report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true_classes, y_pred_classes, target_names=target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true_classes, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 7: Visualize Predictions

Let's see the model's predictions on some test images. **Green** titles mean the prediction is correct, **red** means incorrect.

In [ ]:
# Visualize predictions on test images
fig, axes = plt.subplots(3, 5, figsize=(16, 10))

# Pick random indices from the test set
np.random.seed(42)
random_indices = np.random.choice(len(X_test), 15, replace=False)

for i, ax in enumerate(axes.flat):
    idx = random_indices[i]
    image = X_test[idx].reshape(h, w)
    true_label = target_names[y_true_classes[idx]]
    pred_label = target_names[y_pred_classes[idx]]
    confidence = np.max(y_pred[idx]) * 100

    ax.imshow(image, cmap='gray')
    ax.axis('off')

    # Green if correct, red if wrong
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f"Pred: {pred_label}\n({confidence:.1f}%)", fontsize=9, color=color, fontweight='bold')

plt.suptitle("Model Predictions on Test Set\n(Green = Correct, Red = Incorrect)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

In this notebook we:

1. **Loaded** the Labeled Faces in the Wild (LFW) dataset using scikit-learn
2. **Preprocessed** the images — normalization, reshaping, one-hot encoding, and train/test split
3. **Built** a simple 3-block CNN with Conv2D → BatchNorm → MaxPool layers
4. **Trained** with early stopping to prevent overfitting
5. **Evaluated** using accuracy, classification reports, and a confusion matrix
6. **Visualized** predictions with confidence scores

### Key Takeaways
- **CNNs** are highly effective for image classification tasks like face recognition
- **BatchNormalization** and **Dropout** are important regularization techniques
- **EarlyStopping** prevents overfitting by halting training when validation performance stalls
- Even a **simple 3-layer CNN** can achieve reasonable accuracy on face recognition

### Next Steps to Improve
- Use **Data Augmentation** (rotation, flipping, zooming) to expand the training set
- Try **Transfer Learning** with pretrained models like VGG16 or ResNet
- Increase the dataset size by lowering `min_faces_per_person`
- Experiment with deeper architectures or learning rate schedulers